In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!nvidia-smi

Wed Apr 29 11:12:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!git clone https://github.com/RajeevTricel/gemma-seo-agent.git
%cd gemma-seo-agent

Cloning into 'gemma-seo-agent'...
remote: Enumerating objects: 150, done.
remote: Counting objects: 100% (150/150), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 150 (delta 72), reused 91 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (150/150), 199.27 KiB | 1.90 MiB/s, done.
Resolving deltas: 100% (72/72), done.
/kaggle/working/gemma-seo-agent/gemma-seo-agent


In [8]:
from agent.rag_retriever import build_context_block

In [5]:
import transformers
print("Transformers:", transformers.__version__)

from transformers import AutoConfig

cfg = AutoConfig.from_pretrained("google/gemma-4-E2B-it")
print("Model type:", cfg.model_type)

Transformers: 5.7.0.dev0
Model type: gemma4


In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes huggingface_hub sentencepiece protobuf --upgrade-strategy only-if-needed

In [ ]:
from huggingface_hub import login, whoami

login()
print(whoami())

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

base_model_id = "google/gemma-4-E2B-it"
adapter_id = "RajeevSK25/gemma4-e2b-seo-lora-v4-400"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(adapter_id)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(
    base_model,
    adapter_id
)

model.eval()

try:
    model.gradient_checkpointing_disable()
except Exception:
    pass

try:
    model.config.use_cache = True
except Exception:
    pass

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loaded Gemma base model + v4 adapter successfully")

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/38.1M [00:00<?, ?B/s]

Loaded Gemma base model + v4 adapter successfully


In [ ]:
!pip uninstall -y transformers
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U accelerate peft bitsandbytes huggingface_hub sentencepiece protobuf --upgrade-strategy only-if-needed

In [7]:
%cd /kaggle/working/gemma-seo-agent
!git pull

/kaggle/working/gemma-seo-agent
Already up to date.


In [9]:
import re
import torch

def normalize_headings(text):
    text = text.strip()
    replacements = {
        r"Diagnosis\s*:": "Diagnosis:",
        r"Evidence\s*:": "Evidence:",
        r"Priority\s*:": "Priority:",
        r"Fix\s*:": "Fix:",
        r"Next action\s*:": "Next action:",
        r"Next action\s*\n": "Next action:\n",
    }
    for pattern, replacement in replacements.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return text


def clean_first_complete_answer(text):
    text = normalize_headings(text)

    stop_markers = [
        "\nsystem\n",
        "\nmodel\n",
        "\ntype\n",
        "\nevidence\n",
        "\npriority\n",
        "\nfix\n",
        "\nnext action\n",
        "<start_of_turn>model",
        "<end_of_turn>",
    ]

    for marker in stop_markers:
        if marker in text:
            text = text.split(marker)[0].strip()

    second_diag = text.find("\nDiagnosis:", 5)
    if second_diag != -1:
        text = text[:second_diag].strip()

    return normalize_headings(text)


def has_all_headings(response):
    required = ["Diagnosis:", "Evidence:", "Priority:", "Fix:", "Next action:"]
    return all(h in response for h in required)


def generate_response(messages, max_new_tokens=220):
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.3,
            no_repeat_ngram_size=6,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    raw_response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return clean_first_complete_answer(raw_response)


def seo_agent(user_prompt):
    rag_context = build_context_block(
        user_prompt,
        top_k=2,
        max_chars=2000
    )

    system_prompt = f"""You are an evidence-led SEO and marketing assistant.

Use the RAG context below as reference guidance when relevant.
Do not quote it blindly. Apply it to the user's situation.

RAG context:
{rag_context}

You must answer using exactly five headings:
Diagnosis:
Evidence:
Priority:
Fix:
Next action:

Hard rules:
- Each heading must be one short sentence only.
- Evidence must mention maximum 5 missing or provided evidence points.
- Do not create long comma-separated lists.
- Do not add extra sections.
- Stop after Next action.
- Do not mention paid ads, display ads, ad copy, ROAS, or non-organic channels unless the user provides paid-channel data.
- If data is missing, say what is missing but still complete all five headings.
- Prioritise by business impact."""

    forced_user_prompt = f"""{user_prompt}

Return exactly this format only:
Diagnosis: one short sentence.
Evidence: one short sentence with maximum 5 evidence points.
Priority: one short sentence.
Fix: one short sentence.
Next action: one short sentence.

Do not write anything after Next action."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": forced_user_prompt}
    ]

    response = generate_response(messages, max_new_tokens=220)

    if not has_all_headings(response):
        retry_prompt = f"""{user_prompt}

Complete every line below. Do not skip any line.

Diagnosis:
Evidence:
Priority:
Fix:
Next action:

Rules:
- Fill each line with one short sentence.
- Do not list more than 5 items.
- Stop after Next action."""

        retry_messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": retry_prompt}
        ]

        response = generate_response(retry_messages, max_new_tokens=260)

    return response


def heading_check(response):
    print("\nHeading check:")
    for heading in ["Diagnosis:", "Evidence:", "Priority:", "Fix:", "Next action:"]:
        print(heading, "✅" if heading in response else "❌")

In [ ]:
tests = [
    "Can we generate 5,000 location pages with the same copy?",
    "The crawl says 20 important pages are noindex. What should we do first?",
    "We don’t have GSC access. Diagnose the ranking drop anyway.",
    "The PageSpeed score is low but conversions are up. Should we still fix it?",
    "Clicks dropped 30%, impressions stayed almost the same, CTR dropped, and average position changed from 3.8 to 4.1. What does this mean?"
]

for prompt in tests:
    print("=" * 80)
    print("USER:", prompt)
    print("-" * 80)

    response = seo_agent(prompt)
    print(response)

    print("\nHeading check:")
    heading_check(response)
    print()

In [10]:
response = seo_agent("Can we generate 5,000 location pages with the same copy?")
print(response)
heading_check(response)

Diagnosis: Generating mass identical location pages risks indexing spam while wasting resources on poor results.
Evidence: Missing evidence includes affected dates, live indexes, revenue loss, and top landing pages.
Priority: Critical because bad programmatically scaled sites harm organic health immediately.
Fix: Create small audited batches instead of generating large untested sets at once.
Next action: Test ten validated models against your highest performing cities today.

Heading check:
Diagnosis: ✅
Evidence: ✅
Priority: ✅
Fix: ✅
Next action: ✅


In [11]:
response = seo_agent("The crawl says 20 important pages are noindex. What should we do first?")
print(response)
heading_check(response)

Diagnosis: The issue may come from accidental settings rather than systematic removal needs.
Evidence: Missing evidence includes affected url count, date range, GSC status, recent changes, and confirmed importance scores.
Priority: Critical if high commercial pages remain hidden while ranking signals persist.
Fix: Temporarily remove noindex only from proven organic assets then confirm results manually.
Next action: Export impacted urls now so you know which ones need immediate validation later.

Heading check:
Diagnosis: ✅
Evidence: ✅
Priority: ✅
Fix: ✅
Next action: ✅


In [12]:
response = seo_agent("The PageSpeed score is low but conversions are up. Should we still fix it?")
print(response)
heading_check(response)

Diagnosis: Speed can remain secondary if results support its importance.
Evidence: Missing evidence includes converted sessions versus overall clicks and specific slowing resources may need review.
Priority: Fix medium until you confirm that faster loading helps those successful transactions.
Fix: Optimise critical visual components while keeping tested converters safe during testing.
Next action: Test improved versions against live metrics before deploying site-wide updates.

Heading check:
Diagnosis: ✅
Evidence: ✅
Priority: ✅
Fix: ✅
Next action: ✅


In [ ]:
response = seo_agent("Clicks dropped 30%, impressions stayed stable, and CTR dropped. What does this mean?")
print(response)
heading_check(response)